# EfficientNets

> Pretrained and from scratch EfficientNets Implementation.


In [ ]:
#| default_exp models.vision.efficientnets

In [ ]:
#| hide
from nbdev.showdoc import *  

In [ ]:
#| export
from fastcore.utils import *  
from torch import nn
import torch.nn.functional as F
import torch
from typing import Any, Callable, List, Optional


### Standard EfficientNets

In [ ]:
#| export
def swish(x):
    return x * x.sigmoid()


def drop_connect(x, drop_ratio):
    keep_ratio = 1.0 - drop_ratio
    mask = torch.empty([x.shape[0], 1, 1, 1], dtype=x.dtype, device=x.device)
    mask.bernoulli_(keep_ratio)
    x.div_(keep_ratio)
    x.mul_(mask)
    return x


class SE(nn.Module):
    '''Squeeze-and-Excitation block with Swish.'''

    def __init__(self, in_channels, se_channels):
        super(SE, self).__init__()
        self.se1 = nn.Conv2d(in_channels, se_channels,
                             kernel_size=1, bias=True)
        self.se2 = nn.Conv2d(se_channels, in_channels,
                             kernel_size=1, bias=True)

    def forward(self, x):
        out = F.adaptive_avg_pool2d(x, (1, 1))
        out = swish(self.se1(out))
        out = self.se2(out).sigmoid()
        out = x * out
        return out

In [ ]:
#| export

class Block(nn.Module):
    '''expansion + depthwise + pointwise + squeeze-excitation'''

    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size,
                 stride,
                 expand_ratio=1,
                 se_ratio=0.,
                 drop_rate=0.):
        super(Block, self).__init__()
        self.stride = stride
        self.drop_rate = drop_rate
        self.expand_ratio = expand_ratio

        # Expansion
        channels = expand_ratio * in_channels
        self.conv1 = nn.Conv2d(in_channels,
                               channels,
                               kernel_size=1,
                               stride=1,
                               padding=0,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(channels)

        # Depthwise conv
        self.conv2 = nn.Conv2d(channels,
                               channels,
                               kernel_size=kernel_size,
                               stride=stride,
                               padding=(1 if kernel_size == 3 else 2),
                               groups=channels,
                               bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

        # SE layers
        se_channels = int(in_channels * se_ratio)
        self.se = SE(channels, se_channels)

        # Output
        self.conv3 = nn.Conv2d(channels,
                               out_channels,
                               kernel_size=1,
                               stride=1,
                               padding=0,
                               bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        # Skip connection if in and out shapes are the same (MV-V2 style)
        self.has_skip = (stride == 1) and (in_channels == out_channels)

    def forward(self, x):
        out = x if self.expand_ratio == 1 else swish(self.bn1(self.conv1(x)))
        out = swish(self.bn2(self.conv2(out)))
        out = self.se(out)
        out = self.bn3(self.conv3(out))
        if self.has_skip:
            if self.training and self.drop_rate > 0:
                out = drop_connect(out, self.drop_rate)
            out = out + x
        return out


In [ ]:
#| export
import math

def round_filters(filters, width_mult):
    if not width_mult:
        return filters
    filters *= width_mult
    new_filters = max(8, int(filters + 8 / 2) // 8 * 8)
    if new_filters < 0.9 * filters:
        new_filters += 8
    return int(new_filters)

def round_repeats(repeats, depth_mult):
    if not depth_mult:
        return repeats
    return int(math.ceil(depth_mult * repeats))

class EfficientNetBackbone(nn.Module):
    def __init__(self, width_mult=1.0, depth_mult=1.0, num_classes=10):
        super(EfficientNetBackbone, self).__init__()
        
        # Base B0 configuration
        # (expansion, out_channels, num_blocks, kernel_size, stride)
        self.base_cfg = [
            [1, 16, 1, 3, 1],
            [6, 24, 2, 3, 2],
            [6, 40, 2, 5, 2],
            [6, 80, 3, 3, 2],
            [6, 112, 3, 5, 1],
            [6, 192, 4, 5, 2],
            [6, 320, 1, 3, 1],
        ]
        
        self.drop_connect_rate = 0.2
        # self.dropout_rate = dropout_rate
        
        # Initial Conv
        out_channels = round_filters(32, width_mult)
        self.conv1 = nn.Conv2d(3, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # Build Layers
        self.layers = self._make_layers(out_channels, width_mult, depth_mult)

    def _make_layers(self, in_channels, width_mult, depth_mult):
        layers = []
        # Calculate total blocks for drop_connect schedule
        total_blocks = sum([round_repeats(x[2], depth_mult) for x in self.base_cfg])
        block_idx = 0
        
        for expand, out_c, num_b, k, s in self.base_cfg:
            out_c = round_filters(out_c, width_mult)
            num_b = round_repeats(num_b, depth_mult)
            
            strides = [s] + [1] * (num_b - 1)
            for stride in strides:
                drop_rate = self.drop_connect_rate * block_idx / total_blocks
                layers.append(Block(in_channels, out_c, k, stride, expand, se_ratio=0.25, drop_rate=drop_rate))
                in_channels = out_c
                block_idx += 1
        return nn.Sequential(*layers)

    def forward(self, x):
        out = swish(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(out.size(0), -1)
        return out

In [ ]:
#| export
class EfficientNet(nn.Module):
    def __init__(self, width_mult=1.0, depth_mult=1.0, dropout_rate=0.2, num_classes=10):
        super(EfficientNet, self).__init__()
        
        self.backbone = EfficientNetBackbone(width_mult= width_mult, depth_mult= depth_mult, num_classes= num_classes)
        last_channels = round_filters(320, width_mult)
        self.head = nn.Sequential(
            nn.Dropout(p= dropout_rate),
            nn.Linear(last_channels, num_classes)
        )
        
    def forward(self, x):
        h = self.backbone(x)
        h = h.view(h.size(0), -1)
        logits = self.head(h)
        return logits


In [ ]:
#| export
def EfficientNetB0(num_classes=10):
    return EfficientNet(width_mult=1.0, depth_mult=1.0, dropout_rate=0.2, num_classes=num_classes)

def EfficientNetB1(num_classes=10):
    return EfficientNet(width_mult=1.0, depth_mult=1.1, dropout_rate=0.2, num_classes=num_classes)

def EfficientNetB2(num_classes=10):
    return EfficientNet(width_mult=1.1, depth_mult=1.2, dropout_rate=0.3, num_classes=num_classes)

def EfficientNetB3(num_classes=10):
    return EfficientNet(width_mult=1.2, depth_mult=1.4, dropout_rate=0.3, num_classes=num_classes)

def EfficientNetB4(num_classes=10):
    return EfficientNet(width_mult=1.4, depth_mult=1.8, dropout_rate=0.4, num_classes=num_classes)

def EfficientNetB5(num_classes=10):
    return EfficientNet(width_mult=1.6, depth_mult=2.2, dropout_rate=0.4, num_classes=num_classes)

def EfficientNetB6(num_classes=10):
    return EfficientNet(width_mult=1.8, depth_mult=2.6, dropout_rate=0.5, num_classes=num_classes)
    
def EfficientNetB7(num_classes=10):
    return EfficientNet(width_mult=2.0, depth_mult=3.1, dropout_rate=0.5, num_classes=num_classes)

In [ ]:
#| hide
model = EfficientNetB0(num_classes=10)
model

EfficientNet(
  (backbone): EfficientNetBackbone(
    (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (layers): Sequential(
      (0): Block(
        (conv1): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (se): SE(
          (se1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (se2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
        )
        (conv3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1

### Pretrained EfficientNet

In case you use any of the pretrained resnets, we will need to `upsample` the input images to `224x224` as the pretrained resnets are trained on imagenet which has images of size `224x224`.

In [ ]:
#| export
from torchvision.models import (efficientnet_b0, efficientnet_b1, efficientnet_b2, efficientnet_b3, 
                                efficientnet_b4, efficientnet_b5, efficientnet_b6, efficientnet_b7)

In [ ]:
#| hide
pretrained_model = efficientnet_b0(weights='IMAGENET1K_V1')
pretrained_model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
#| export
class PretrainedEfficient(nn.Module):

    def __init__(self, backbone, in_features, num_classes=10):
        super(PretrainedEfficient, self).__init__()
        
        self.backbone = backbone
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        h = self.backbone(x)
        h = h.view(h.size(0), -1)
        logits = self.head(h)
        return logits

In [ ]:
#| export
def efficientx(x= 0, num_classes= 1000):
    model_dict = {
            "efficientnet_b0": efficientnet_b0,
            "efficientnet_b1": efficientnet_b1,
            "efficientnet_b2": efficientnet_b2,
            "efficientnet_b3": efficientnet_b3,
            "efficientnet_b4": efficientnet_b4,
            "efficientnet_b5": efficientnet_b5,
            "efficientnet_b6": efficientnet_b6,
            "efficientnet_b7": efficientnet_b7
    }
    
    model =  model_dict[f"efficientnet_b{x}"](weights= "IMAGENET1K_V1")
    l = []
    for name, m in model.named_children():
        if name in ['features', 'avgpool']:
            l.append(m)
    backbone = nn.Sequential(*l)

    model = PretrainedEfficient(backbone= backbone, in_features= model.classifier[1].in_features, num_classes= num_classes)
    return model

In [ ]:
#| hide
model = efficientx(x= 0, num_classes= 10)
model


PretrainedEfficient(
  (backbone): Sequential(
    (0): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (scale_a

## Unified API for EfficientNets

In [ ]:
#| export
from typing import Union, Literal
import torch.nn as nn
from fedai.models.vision.registery import register_model

The API allows to create different variants of EfficientNets with or without pretrained weights using a single function `create_model`. For example:

```python
 model = make_efficientnet(0)  # EfficientNet-B0
 model = make_efficientnet('b3')  # EfficientNet-B3
 model = make_efficientnet('efficientnet_b5')  # EfficientNet-B5
```

In [ ]:
#| export
@register_model(
    name='efficientnet',
    variants=['b0', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7']
)
def make_efficientnet(
    variant: Union[int, str] = 0,
    pretrained: bool = False,
    num_classes: int = 10,
) -> nn.Module:
    """
    Create an EfficientNet model.
    
    Args:
        variant: Model variant, either int (0-7) or string ('b0'-'b7', 'efficientnet_b0'-'efficientnet_b7')
        num_classes: Number of output classes
        pretrained: Whether to load pretrained weights
    
    Examples:
        >>> model = make_efficientnet(0)  # EfficientNet-B0
        >>> model = make_efficientnet('b3')  # EfficientNet-B3
        >>> model = make_efficientnet('efficientnet_b5')  # EfficientNet-B5
    """
    # Normalize variant to standard format
    if isinstance(variant, int):
        if not 0 <= variant <= 7:
            raise ValueError(f"EfficientNet variant must be 0-7, got {variant}")
        
        variant_key = f"efficientnet_b{variant}"
    elif isinstance(variant, str):
        # Handle both "b3" and "efficientnet_b3"
        if variant.startswith('b') and variant[1:].isdigit():
            variant_key = f"efficientnet_{variant}"
        elif variant.startswith('efficientnet_b'):
            variant_key = variant
        else:
            raise ValueError(f"Invalid EfficientNet variant: {variant}")
    else:
        raise TypeError(f"variant must be int or str, got {type(variant)}")
    
    if pretrained:
        # Extract just the number for pretrained loader
        b_num = int(variant_key.split('_b')[1])
        return efficientx(b_num, num_classes)
    
    model_dict = {
        "efficientnet_b0": EfficientNetB0,
        "efficientnet_b1": EfficientNetB1,
        "efficientnet_b2": EfficientNetB2,
        "efficientnet_b3": EfficientNetB3,
        "efficientnet_b4": EfficientNetB4,
        "efficientnet_b5": EfficientNetB5,
        "efficientnet_b6": EfficientNetB6,
        "efficientnet_b7": EfficientNetB7
    }
    
    return model_dict[variant_key](num_classes=num_classes)

In [ ]:
#| hide
model = make_efficientnet(0)  # EfficientNet-B0
model = make_efficientnet('b0')  # EfficientNet-B0
model = make_efficientnet('efficientnet_b0')  # EfficientNet-B0
model = make_efficientnet(0, pretrained= True)  # EfficientNet-B0
model = make_efficientnet('b0', pretrained= True)  # EfficientNet-B0
model = make_efficientnet('efficientnet_b0', pretrained= True, num_classes= 10)


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /home/ahmed/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:03<00:00, 12.8MB/s]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()